# 🔙 Backtracking -- Runnable Notebook

Companion to [`README.md`](README.md) and the interactive
[`21_backtracking_lesson.html`](21_backtracking_lesson.html).

Every backtracking problem in **Blind 75** and **NeetCode 150**, built from one template
(**choose → explore → un-choose**). Run each cell top to bottom -- every function prints its result and
`assert`s the expected answer, and two cells deliberately show the classic bugs so you can see *why* the
un-choose step and the `path[:]` copy matter.

## 1. The template -- Subsets (LC 78), two ways

**Problem in plain language:** you're given a list of distinct numbers. Return **every possible subset**
of it (the "power set") — including the empty subset and the full list itself. The order of numbers
inside a subset doesn't matter, and you shouldn't return the same subset twice.
Example: `[1, 2]` → `[[], [1], [2], [1, 2]]`.

**Include / exclude:** a binary decision at every index -- the recursion tree has exactly `2^n` leaves.
**Start-index loop:** record every node on entry, and only ever move *forward* so `[1, 2]` and `[2, 1]`
can't both appear. Both produce the same 8 subsets of `[1, 2, 3]`, in different orders.

In [1]:
def subsets_include_exclude(nums):
    out, path = [], []
    def backtrack(i):
        if i == len(nums):                     # base case: decided every element
            out.append(path[:])                # record a COPY (see the bug demo below)
            return
        path.append(nums[i]); 
        backtrack(i + 1); 
        
        path.pop()   # INCLUDE nums[i], then un-choose

        backtrack(i + 1)                                      # EXCLUDE nums[i]
    backtrack(0)
    return out


def subsets(nums):
    """Start-index loop: every prefix on the way down is itself a subset."""
    out, path = [], []
    def backtrack(start):
        out.append(path[:])                    # record on ENTRY, not just at leaves
        for i in range(start, len(nums)):
            path.append(nums[i])               # CHOOSE
            backtrack(i + 1)                   # EXPLORE -- only indices AFTER i
            path.pop()                         # UN-CHOOSE
    backtrack(0)
    return out


a = subsets_include_exclude([1, 2, 3])
b = subsets([1, 2, 3])
print("include/exclude order:", a)
print("start-index order    :", b)
assert len(a) == len(b) == 8                                   # 2^3 subsets
assert sorted(map(sorted, a)) == sorted(map(sorted, b))        # same subsets, different order
assert [] in a and [1, 2, 3] in a

include/exclude order: [[1, 2, 3], [1, 2], [1, 3], [1], [2, 3], [2], [3], []]
start-index order    : [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]


### Step-by-step traces: why the two orders differ

Both walk `nums = [1, 2, 3]`, but they decide *when* to record and *how* to branch differently.

#### `subsets_include_exclude` — record only at leaves

Each call to `backtrack(i)` does two things at index `i`: first explore **with** `nums[i]` in the path,
then pop it and explore **without** it — both branches call `backtrack(i + 1)`.

| Step | Action | `path` after | `out` after |
|---|---|---|---|
| 1 | `backtrack(0)` starts | `[]` | `[]` |
| 2 | CHOOSE `1` | `[1]` | `[]` |
| 3 | `backtrack(1)` → CHOOSE `2` | `[1,2]` | `[]` |
| 4 | `backtrack(2)` → CHOOSE `3` | `[1,2,3]` | `[]` |
| 5 | `backtrack(3)`: `i==3` → **RECORD** | `[1,2,3]` | `[[1,2,3]]` |
| 6 | UN-CHOOSE (pop `3`) | `[1,2]` | `[[1,2,3]]` |
| 7 | `backtrack(3)` again (exclude `3`): `i==3` → **RECORD** | `[1,2]` | `[..., [1,2]]` |
| 8 | back in `backtrack(1)`: UN-CHOOSE (pop `2`) | `[1]` | `[..., [1,2]]` |
| 9 | `backtrack(2)` again (exclude `2`) → CHOOSE `3` | `[1,3]` | `[..., [1,2]]` |
| 10 | `backtrack(3)`: **RECORD** | `[1,3]` | `[..., [1,3]]` |
| 11 | UN-CHOOSE (pop `3`) | `[1]` | `[..., [1,3]]` |
| 12 | `backtrack(3)` again (exclude `3`): **RECORD** | `[1]` | `[..., [1]]` |
| 13 | back in `backtrack(0)`: UN-CHOOSE (pop `1`) | `[]` | `[..., [1]]` |
| 14 | `backtrack(1)` again (exclude `1`) → CHOOSE `2` | `[2]` | `[..., [1]]` |
| 15 | `backtrack(2)` → CHOOSE `3` | `[2,3]` | `[..., [1]]` |
| 16 | `backtrack(3)`: **RECORD** | `[2,3]` | `[..., [2,3]]` |
| 17 | UN-CHOOSE (pop `3`) | `[2]` | `[..., [2,3]]` |
| 18 | `backtrack(3)` again: **RECORD** | `[2]` | `[..., [2]]` |
| 19 | UN-CHOOSE (pop `2`) | `[]` | `[..., [2]]` |
| 20 | `backtrack(2)` again (exclude `2`) → CHOOSE `3` | `[3]` | `[..., [2]]` |
| 21 | `backtrack(3)`: **RECORD** | `[3]` | `[..., [3]]` |
| 22 | UN-CHOOSE (pop `3`) | `[]` | `[..., [3]]` |
| 23 | `backtrack(3)` again (exclude `3`): **RECORD** | `[]` | `[..., []]` |

Final `out`: `[[1,2,3], [1,2], [1,3], [1], [2,3], [2], [3], []]` — only ever recorded at `i==3` (a leaf).

#### `subsets` — record on every entry (start-index loop)

Every call **records immediately on entry**, then loops forward from `start`.

| Step | Action | `path` after | `out` after |
|---|---|---|---|
| 1 | `backtrack(0)` entered → **RECORD** | `[]` | `[[]]` |
| 2 | `i=0`: CHOOSE `1` | `[1]` | `[[]]` |
| 3 | `backtrack(1)` entered → **RECORD** | `[1]` | `[..., [1]]` |
| 4 | `i=1`: CHOOSE `2` | `[1,2]` | `[..., [1]]` |
| 5 | `backtrack(2)` entered → **RECORD** | `[1,2]` | `[..., [1,2]]` |
| 6 | `i=2`: CHOOSE `3` | `[1,2,3]` | `[..., [1,2]]` |
| 7 | `backtrack(3)` entered → **RECORD** | `[1,2,3]` | `[..., [1,2,3]]` |
| 8 | `backtrack(3)`'s loop (`range(3,3)`) is empty → return | `[1,2,3]` | same |
| 9 | back in `backtrack(2)`: UN-CHOOSE (pop `3`) | `[1,2]` | same |
| 10 | `backtrack(2)`'s loop done (only `i=2`) → return | `[1,2]` | same |
| 11 | back in `backtrack(1)`: UN-CHOOSE (pop `2`) | `[1]` | same |
| 12 | `i=2`: CHOOSE `3` | `[1,3]` | same |
| 13 | `backtrack(3)` entered → **RECORD** | `[1,3]` | `[..., [1,3]]` |
| 14 | UN-CHOOSE (pop `3`) | `[1]` | same |
| 15 | `backtrack(1)`'s loop done → return | `[1]` | same |
| 16 | back in `backtrack(0)`: UN-CHOOSE (pop `1`) | `[]` | same |
| 17 | `i=1`: CHOOSE `2` | `[2]` | same |
| 18 | `backtrack(2)` entered → **RECORD** | `[2]` | `[..., [2]]` |
| 19 | `i=2`: CHOOSE `3` | `[2,3]` | same |
| 20 | `backtrack(3)` entered → **RECORD** | `[2,3]` | `[..., [2,3]]` |
| 21 | UN-CHOOSE (pop `3`) | `[2]` | same |
| 22 | UN-CHOOSE (pop `2`) | `[]` | same |
| 23 | `i=2`: CHOOSE `3` | `[3]` | same |
| 24 | `backtrack(3)` entered → **RECORD** | `[3]` | `[..., [3]]` |
| 25 | UN-CHOOSE (pop `3`) | `[]` | same |

Final `out`: `[[], [1], [1,2], [1,2,3], [1,3], [2], [2,3], [3]]` — recorded on **every** call, including
the very first empty one.

**The key difference the traces show:** `subsets_include_exclude` only ever writes to `out` when it hits
`i==3` (a fixed-depth leaf — 8 leaves for 3 binary choices); `subsets` writes to `out` on literally every
call — 8 calls total here, one of which happens to be the empty path at the very start.

### The two classic bugs, on purpose

1. **Missing un-choose** -- without `path.pop()`, every sibling inherits the previous sibling's choice.
2. **Recording a reference** -- `out.append(path)` stores the *same* list eight times; by the end it's empty.

In [2]:
def subsets_missing_pop(nums):
    out, path = [], []
    def backtrack(start):
        out.append(path[:])
        for i in range(start, len(nums)):
            path.append(nums[i])
            backtrack(i + 1)
            # path.pop()   <-- forgotten
    backtrack(0)
    return out

def subsets_no_copy(nums):
    out, path = [], []
    def backtrack(start):
        out.append(path)                       # reference, not a copy
        for i in range(start, len(nums)):
            path.append(nums[i]); backtrack(i + 1); path.pop()
    backtrack(0)
    return out

bug1 = subsets_missing_pop([1, 2, 3])
bug2 = subsets_no_copy([1, 2, 3])
print("missing pop  :", bug1)
print("no copy      :", bug2)
assert len(bug1) == 8 and [1, 3] not in bug1      # right count, wrong contents: [1,3] never appears
assert bug2 == [[]] * 8                            # eight references to one list, which is empty at the end

missing pop  : [[], [1], [1, 2], [1, 2, 3], [1, 2, 3, 3], [1, 2, 3, 3, 2], [1, 2, 3, 3, 2, 3], [1, 2, 3, 3, 2, 3, 3]]
no copy      : [[], [], [], [], [], [], [], []]


## 2. Subsets II -- duplicates in the input (LC 90)

**Problem in plain language:** same task as Subsets (LC 78) -- return every possible subset -- but now
the input list can contain **duplicate numbers** (e.g. `[1, 2, 2]`). Return every **distinct** subset
only; don't output the same subset twice just because a number was repeated in the input.
Example: `[1, 2, 2]` → `[[], [1], [1,2], [1,2,2], [2], [2,2]]` (not 8 subsets, only 6 distinct ones).

Sort so equal values sit together, then skip a value when it equals its left neighbour **and** that neighbour
was a *sibling* we already finished (`i > start`), not an ancestor on the current path.

In [3]:
def subsets_with_dup(nums):
    nums = sorted(nums)
    out, path = [], []
    def backtrack(start):
        out.append(path[:])
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                       # the sibling just before us already built this branch
            path.append(nums[i]); 
            
            backtrack(i + 1); 
            path.pop()
    backtrack(0)
    return out


res = subsets_with_dup([1, 2, 2])
print("subsets of [1,2,2]:", res)
assert res == [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
assert len(res) == len({tuple(s) for s in res})   # no duplicates

# Without the skip you get duplicate subsets -- the plain version treats the two 2s as different elements.
plain = subsets([1, 2, 2])
print("without the skip:", len(plain), "subsets,", len({tuple(s) for s in plain}), "distinct")
assert len(plain) == 8 and len({tuple(s) for s in plain}) == 6

subsets of [1,2,2]: [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
without the skip: 8 subsets, 6 distinct


### Same idea, your preferred shape -- include/exclude

`subsets_with_dup` above uses a start-index loop. Here's the binary choose/skip version, same shape as
`subsets_include_exclude` from section 1: at each index `i`, either **include** `nums[i]` and move to
`i + 1`, or **exclude** it -- but excluding it means skipping *every* remaining copy of that value too,
because "exclude the first 2" and "exclude the second 2" would otherwise produce the same subset twice.

In [4]:
def subsets_with_dup_include_exclude(nums):
    nums = sorted(nums)
    n = len(nums)
    out, path = [], []
    def backtrack(i):
        if i == n:
            out.append(path[:]); return
        path.append(nums[i])                              # INCLUDE nums[i]
        backtrack(i + 1)
        path.pop()
        j = i + 1
        while j < n and nums[j] == nums[i]:                # EXCLUDE nums[i] -- and every duplicate of it,
            j += 1                                         # since skipping the first is the same choice
        backtrack(j)                                       # as skipping the second, third, ...
    backtrack(0)
    return out


ie = subsets_with_dup_include_exclude([1, 2, 2])
print("include/exclude, subsets of [1,2,2]:", ie)
assert sorted(map(sorted, ie)) == sorted(map(sorted, res))  # same 6 subsets, different order
assert sorted(map(sorted, subsets_with_dup_include_exclude([4, 4, 4, 1, 4]))) == \
       sorted(map(sorted, subsets_with_dup([4, 4, 4, 1, 4])))

include/exclude, subsets of [1,2,2]: [[1, 2, 2], [1, 2], [1], [2, 2], [2], []]


## 3. Combinations & Combination Sum (LC 77, LC 39, LC 40)

**Problem in plain language — three separate problems here:**
- **Combinations (LC 77):** given numbers `n` and `k`, return every way to **choose `k` numbers out of
  `1..n`**, where order doesn't matter (`{1,2}` and `{2,1}` count as the same choice, so only one is kept).
- **Combination Sum (LC 39):** given a list of candidate numbers and a `target`, return every combination
  of candidates that **adds up exactly to `target`**. You're allowed to reuse the same candidate as many
  times as you like.
- **Combination Sum II (LC 40):** same as Combination Sum, but each number in the input list can be used
  **at most once** (even if it appears more than once in the input), and duplicate combinations in the
  output aren't allowed.

Same start-index loop, different base case: record when the path reaches length `k` or sum `target`.
- `backtrack(i, ...)` lets a candidate be **reused**; `backtrack(i + 1, ...)` uses each element **once**.
- With sorted candidates, `break` the moment one is too big -- everything after it is too.

In [5]:
def combine(n, k):                                    # LC 77
    out, path = [], []
    def backtrack(start):
        if len(path) == k:
            out.append(path[:]); return
        for i in range(start, n + 1):
            path.append(i); backtrack(i + 1); path.pop()
    backtrack(1)
    return out


def combination_sum(candidates, target):                 # LC 39 -- unlimited reuse
    candidates = sorted(candidates)
    out, path = [], []
    def backtrack(start, remaining):
        if remaining == 0:
            out.append(path[:]); return
        for i in range(start, len(candidates)):
            if candidates[i] > remaining:
                break                                    # PRUNE: sorted, so nothing later fits either
            path.append(candidates[i])
            backtrack(i, remaining - candidates[i])      # `i`, not `i + 1`: may reuse this candidate
            path.pop()
    backtrack(0, target)
    return out


def combination_sum2(candidates, target):                # LC 40 -- each element once, no duplicate combos
    candidates = sorted(candidates)
    out, path = [], []
    def backtrack(start, remaining):
        if remaining == 0:
            out.append(path[:]); return
        for i in range(start, len(candidates)):
            if candidates[i] > remaining:
                break
            if i > start and candidates[i] == candidates[i - 1]:
                continue                                 # sort + skip sibling duplicate (same as Subsets II)
            path.append(candidates[i])
            backtrack(i + 1, remaining - candidates[i])  # `i + 1`: each element at most once
            path.pop()
    backtrack(0, target)
    return out


c = combine(4, 2)
print("combine(4, 2):", c)
assert c == [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]

cs = combination_sum([2, 3, 6, 7], 7)
print("combination_sum([2,3,6,7], 7):", cs)
assert cs == [[2, 2, 3], [7]]

cs2 = combination_sum2([10, 1, 2, 7, 6, 1, 5], 8)
print("combination_sum2([10,1,2,7,6,1,5], 8):", cs2)
assert cs2 == [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]

combine(4, 2): [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]
combination_sum([2,3,6,7], 7): [[2, 2, 3], [7]]
combination_sum2([10,1,2,7,6,1,5], 8): [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]


### The same three problems, include/exclude style

Same choose → explore → un-choose recursion, but structured as a **binary decision per candidate**
instead of a start-index loop -- the shape from section 1's `subsets_include_exclude`. Each function below
solves the exact same problem as its start-index twin above; the asserts confirm they agree.

- **Combinations:** at position `num`, either include it and move to `num + 1`, or skip it and move to
  `num + 1` anyway -- the only difference from Subsets' include/exclude is the base case (`len(path) == k`,
  not `i == n`).
- **Combination Sum:** at index `i`, either take `candidates[i]` (stay at `i` -- reuse allowed) or skip it
  entirely (move to `i + 1`). No duplicate handling needed -- the input has no repeats.
- **Combination Sum II:** at index `i`, either take one copy of `candidates[i]` (move to `i + 1` -- no
  reuse) or exclude it -- and, exactly like Subsets II above, skip every remaining copy of that same value
  too when excluding, since excluding the first `1` and excluding the second `1` are the same decision.

In [6]:
def combine_include_exclude(n, k):                           # LC 77
    out, path = [], []
    def backtrack(num):
        if len(path) == k:
            out.append(path[:]); return
        if num > n:
            return
        path.append(num)                                      # INCLUDE num
        backtrack(num + 1)
        path.pop()
        backtrack(num + 1)                                     # EXCLUDE num
    backtrack(1)
    return out


def combination_sum_include_exclude(candidates, target):        # LC 39 -- unlimited reuse
    out, path = [], []
    def backtrack(i, remaining):
        if remaining == 0:
            out.append(path[:]); return
        if i == len(candidates) or remaining < 0:
            return
        path.append(candidates[i])                             # INCLUDE candidates[i] (stay at i: may reuse)
        backtrack(i, remaining - candidates[i])
        path.pop()
        backtrack(i + 1, remaining)                             # EXCLUDE candidates[i] entirely
    backtrack(0, target)
    return out


def combination_sum2_include_exclude(candidates, target):       # LC 40 -- each element once, no dup combos
    candidates = sorted(candidates)
    n = len(candidates)
    out, path = [], []
    def backtrack(i, remaining):
        if remaining == 0:
            out.append(path[:]); return
        if i == n or remaining < 0:
            return
        path.append(candidates[i])                             # INCLUDE one copy (move to i+1: no reuse)
        backtrack(i + 1, remaining - candidates[i])
        path.pop()
        j = i + 1
        while j < n and candidates[j] == candidates[i]:          # EXCLUDE candidates[i] -- and every
            j += 1                                                # duplicate of it, same trick as Subsets II
        backtrack(j, remaining)
    backtrack(0, target)
    return out


c_ie = combine_include_exclude(4, 2)
print("combine_include_exclude(4, 2):", c_ie)
assert sorted(c_ie) == sorted(c)                                 # same 6 combinations as combine(4, 2)

cs_ie = combination_sum_include_exclude([2, 3, 6, 7], 7)
print("combination_sum_include_exclude([2,3,6,7], 7):", cs_ie)
assert sorted(map(sorted, cs_ie)) == sorted(map(sorted, cs))     # same answers as combination_sum

cs2_ie = combination_sum2_include_exclude([10, 1, 2, 7, 6, 1, 5], 8)
print("combination_sum2_include_exclude(...):", cs2_ie)
assert sorted(map(sorted, cs2_ie)) == sorted(map(sorted, cs2))   # same answers as combination_sum2

combine_include_exclude(4, 2): [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]
combination_sum_include_exclude([2,3,6,7], 7): [[2, 2, 3], [7]]
combination_sum2_include_exclude(...): [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]


### Step-by-step: what this cell actually does

Both functions share one recursion, `backtrack(start, remaining)`, over the **sorted** candidates: loop
forward from `start`, `break` the moment a candidate exceeds `remaining` (sorted ⇒ everything after is
too big), choose it, recurse, then un-choose. They differ in exactly two lines — whether the recursive
call reuses index `i` or advances to `i + 1`, and whether there's a duplicate-sibling skip. (`combine`,
the third function in this cell, is plain LC 77 and isn't traced here.)

#### `combination_sum` — reuse allowed, record when `remaining == 0`

1. Sort candidates.
2. `backtrack(start, remaining)`: if `remaining == 0`, record `path[:]` and return.
3. Loop `i` from `start`; `break` if `candidates[i] > remaining`.
4. CHOOSE `candidates[i]`; recurse **`backtrack(i, remaining - candidates[i])`** (same `i` — may reuse it); UN-CHOOSE (pop).

Demo: `candidates = [2, 3, 6, 7]` (already sorted), `target = 7`.

| Step | Action | `path` after | `remaining` after | `res` after |
|---|---|---|---|---|
| 1 | `backtrack(0, 7)` entered | `[]` | 7 | `[]` |
| 2 | `i=0`: CHOOSE `2` | `[2]` | 5 | `[]` |
| 3 | `backtrack(0, 5)` → `i=0`: CHOOSE `2` | `[2, 2]` | 3 | `[]` |
| 4 | `backtrack(0, 3)` → `i=0`: CHOOSE `2` | `[2, 2, 2]` | 1 | `[]` |
| 5 | `backtrack(0, 1)` → `i=0`: `2 > 1` → **BREAK** | `[2, 2, 2]` | 1 | `[]` |
| 6 | UN-CHOOSE (pop `2`) | `[2, 2]` | 3 | `[]` |
| 7 | `i=1`: `3 ≤ 3` → CHOOSE `3` | `[2, 2, 3]` | 0 | `[]` |
| 8 | `backtrack(1, 0)`: `remaining==0` → **RECORD** | `[2, 2, 3]` | 0 | `[[2,2,3]]` |
| 9 | UN-CHOOSE (pop `3`) | `[2, 2]` | 3 | `[[2,2,3]]` |
| 10 | `i=2`: `6 > 3` → **BREAK** (ends `backtrack(0,3)`) | `[2, 2]` | 3 | `[[2,2,3]]` |
| 11 | UN-CHOOSE (pop `2`) | `[2]` | 5 | `[[2,2,3]]` |
| 12 | `i=1`: `3 ≤ 5` → CHOOSE `3` | `[2, 3]` | 2 | `[[2,2,3]]` |
| 13 | `backtrack(1, 2)` → `i=1`: `3 > 2` → **BREAK** | `[2, 3]` | 2 | `[[2,2,3]]` |
| 14 | UN-CHOOSE (pop `3`) | `[2]` | 5 | `[[2,2,3]]` |
| 15 | `i=2`: `6 > 5` → **BREAK** (ends `backtrack(0,5)`) | `[2]` | 5 | `[[2,2,3]]` |
| 16 | UN-CHOOSE (pop `2`) | `[]` | 7 | `[[2,2,3]]` |
| 17 | `i=1`: `3 ≤ 7` → CHOOSE `3` | `[3]` | 4 | `[[2,2,3]]` |
| 18 | `backtrack(1, 4)` → `i=1`: `3 ≤ 4` → CHOOSE `3` | `[3, 3]` | 1 | `[[2,2,3]]` |
| 19 | `backtrack(1, 1)` → `i=1`: `3 > 1` → **BREAK** | `[3, 3]` | 1 | `[[2,2,3]]` |
| 20 | UN-CHOOSE (pop `3`) | `[3]` | 4 | `[[2,2,3]]` |
| 21 | `i=2`: `6 > 4` → **BREAK** (ends `backtrack(1,4)`) | `[3]` | 4 | `[[2,2,3]]` |
| 22 | UN-CHOOSE (pop `3`) | `[]` | 7 | `[[2,2,3]]` |
| 23 | `i=2`: `6 ≤ 7` → CHOOSE `6` | `[6]` | 1 | `[[2,2,3]]` |
| 24 | `backtrack(2, 1)` → `i=2`: `6 > 1` → **BREAK** | `[6]` | 1 | `[[2,2,3]]` |
| 25 | UN-CHOOSE (pop `6`) | `[]` | 7 | `[[2,2,3]]` |
| 26 | `i=3`: `7 ≤ 7` → CHOOSE `7` | `[7]` | 0 | `[[2,2,3]]` |
| 27 | `backtrack(3, 0)`: `remaining==0` → **RECORD** | `[7]` | 0 | `[[2,2,3], [7]]` |
| 28 | UN-CHOOSE (pop `7`) | `[]` | 7 | `[[2,2,3], [7]]` |
| 29 | loop out of candidates (`i=4`) → `backtrack(0,7)` returns | `[]` | 7 | `[[2,2,3], [7]]` |

Final `res = [[2, 2, 3], [7]]` — matches `assert cs == [[2, 2, 3], [7]]`.

#### `combination_sum2` — each array slot once, duplicate siblings skipped

1. Sort candidates (repeats now sit next to each other).
2. `backtrack(start, remaining)`: if `remaining == 0`, record `path[:]` and return.
3. Loop `i` from `start`; `break` if `candidates[i] > remaining`.
4. **Skip** if `i > start and candidates[i] == candidates[i - 1]` (a sibling equal to the one just tried at this same call — already fully explored).
5. CHOOSE `candidates[i]`; recurse **`backtrack(i + 1, remaining - candidates[i])`** (next index — no reuse); UN-CHOOSE (pop).

Demo: `candidates = [10, 1, 2, 7, 6, 1, 5]` sorted to `[1, 1, 2, 5, 6, 7, 10]` (indices `0..6`), `target = 8`.

| Step | Action | `path` after | `remaining` after | `res` after |
|---|---|---|---|---|
| 1 | `backtrack(0, 8)` entered | `[]` | 8 | `[]` |
| 2 | `i=0`: CHOOSE `1` | `[1]` | 7 | `[]` |
| 3 | `backtrack(1, 7)` → `i=1` (`i==start`, no skip check): CHOOSE `1` | `[1, 1]` | 6 | `[]` |
| 4 | `backtrack(2, 6)` → `i=2`: CHOOSE `2` | `[1, 1, 2]` | 4 | `[]` |
| 5 | `backtrack(3, 4)` → `i=3`: `5 > 4` → **BREAK** | `[1, 1, 2]` | 4 | `[]` |
| 6 | UN-CHOOSE (pop `2`) | `[1, 1]` | 6 | `[]` |
| 7 | `i=3`: `5 ≤ 6`, `5 ≠ candidates[2]=2` → CHOOSE `5` | `[1, 1, 5]` | 1 | `[]` |
| 8 | `backtrack(4, 1)` → `i=4`: `6 > 1` → **BREAK** | `[1, 1, 5]` | 1 | `[]` |
| 9 | UN-CHOOSE (pop `5`) | `[1, 1]` | 6 | `[]` |
| 10 | `i=4`: `6 ≤ 6`, `6 ≠ candidates[3]=5` → CHOOSE `6` | `[1, 1, 6]` | 0 | `[]` |
| 11 | `backtrack(5, 0)`: `remaining==0` → **RECORD** | `[1, 1, 6]` | 0 | `[[1,1,6]]` |
| 12 | UN-CHOOSE (pop `6`) | `[1, 1]` | 6 | `[[1,1,6]]` |
| 13 | `i=5`: `7 > 6` → **BREAK** (ends `backtrack(2,6)`) | `[1, 1]` | 6 | `[[1,1,6]]` |
| 14 | UN-CHOOSE (pop second `1`) | `[1]` | 7 | `[[1,1,6]]` |
| 15 | `i=2`: `2 ≤ 7`, `2 ≠ candidates[1]=1` → CHOOSE `2` | `[1, 2]` | 5 | `[[1,1,6]]` |
| 16 | `backtrack(3, 5)` → `i=3`: `5 ≤ 5` → CHOOSE `5` | `[1, 2, 5]` | 0 | `[[1,1,6]]` |
| 17 | `backtrack(4, 0)`: `remaining==0` → **RECORD** | `[1, 2, 5]` | 0 | `[[1,1,6],[1,2,5]]` |
| 18 | UN-CHOOSE (pop `5`) | `[1, 2]` | 5 | `[[1,1,6],[1,2,5]]` |
| 19 | `i=4`: `6 > 5` → **BREAK** (ends `backtrack(3,5)`) | `[1, 2]` | 5 | `[[1,1,6],[1,2,5]]` |
| 20 | UN-CHOOSE (pop `2`) | `[1]` | 7 | `[[1,1,6],[1,2,5]]` |
| 21 | `i=3`: `5 ≤ 7`, `5 ≠ candidates[2]=2` → CHOOSE `5` | `[1, 5]` | 2 | `[[1,1,6],[1,2,5]]` |
| 22 | `backtrack(4, 2)` → `i=4`: `6 > 2` → **BREAK** | `[1, 5]` | 2 | `[[1,1,6],[1,2,5]]` |
| 23 | UN-CHOOSE (pop `5`) | `[1]` | 7 | `[[1,1,6],[1,2,5]]` |
| 24 | `i=4`: `6 ≤ 7`, `6 ≠ candidates[3]=5` → CHOOSE `6` | `[1, 6]` | 1 | `[[1,1,6],[1,2,5]]` |
| 25 | `backtrack(5, 1)` → `i=5`: `7 > 1` → **BREAK** | `[1, 6]` | 1 | `[[1,1,6],[1,2,5]]` |
| 26 | UN-CHOOSE (pop `6`) | `[1]` | 7 | `[[1,1,6],[1,2,5]]` |
| 27 | `i=5`: `7 ≤ 7`, `7 ≠ candidates[4]=6` → CHOOSE `7` | `[1, 7]` | 0 | `[[1,1,6],[1,2,5]]` |
| 28 | `backtrack(6, 0)`: `remaining==0` → **RECORD** | `[1, 7]` | 0 | `[[1,1,6],[1,2,5],[1,7]]` |
| 29 | UN-CHOOSE (pop `7`) | `[1]` | 7 | `[[1,1,6],[1,2,5],[1,7]]` |
| 30 | `i=6`: `10 > 7` → **BREAK** (ends `backtrack(1,7)`) | `[1]` | 7 | `[[1,1,6],[1,2,5],[1,7]]` |
| 31 | UN-CHOOSE (pop first `1`) | `[]` | 8 | `[[1,1,6],[1,2,5],[1,7]]` |
| 32 | `i=1`: `i(1) > start(0)` and `candidates[1]=1 == candidates[0]=1` → **SKIP** (no choose) | `[]` | 8 | `[[1,1,6],[1,2,5],[1,7]]` |
| 33 | `i=2`: `2 ≤ 8`, `2 ≠ candidates[1]=1` → CHOOSE `2` | `[2]` | 6 | `[[1,1,6],[1,2,5],[1,7]]` |
| 34 | `backtrack(3, 6)` → `i=3`: `5 ≤ 6` → CHOOSE `5` | `[2, 5]` | 1 | `[[1,1,6],[1,2,5],[1,7]]` |
| 35 | `backtrack(4, 1)` → `i=4`: `6 > 1` → **BREAK** | `[2, 5]` | 1 | `[[1,1,6],[1,2,5],[1,7]]` |
| 36 | UN-CHOOSE (pop `5`) | `[2]` | 6 | `[[1,1,6],[1,2,5],[1,7]]` |
| 37 | `i=4`: `6 ≤ 6`, `6 ≠ candidates[3]=5` → CHOOSE `6` | `[2, 6]` | 0 | `[[1,1,6],[1,2,5],[1,7]]` |
| 38 | `backtrack(5, 0)`: `remaining==0` → **RECORD** | `[2, 6]` | 0 | `[[1,1,6],[1,2,5],[1,7],[2,6]]` |
| 39 | UN-CHOOSE (pop `6`) | `[2]` | 6 | `[[1,1,6],[1,2,5],[1,7],[2,6]]` |
| 40 | `i=5`: `7 > 6` → **BREAK** (ends `backtrack(3,6)`) | `[2]` | 6 | `[[1,1,6],[1,2,5],[1,7],[2,6]]` |
| 41 | UN-CHOOSE (pop `2`) | `[]` | 8 | `[[1,1,6],[1,2,5],[1,7],[2,6]]` |
| 42 | `i=3`: `5 ≤ 8`, `5 ≠ candidates[2]=2` → CHOOSE `5` | `[5]` | 3 | `[[1,1,6],[1,2,5],[1,7],[2,6]]` |
| 43 | `backtrack(4, 3)` → `i=4`: `6 > 3` → **BREAK** | `[5]` | 3 | `[[1,1,6],[1,2,5],[1,7],[2,6]]` |
| 44 | UN-CHOOSE (pop `5`) | `[]` | 8 | `[[1,1,6],[1,2,5],[1,7],[2,6]]` |
| 45 | `i=4`: `6 ≤ 8`, `6 ≠ candidates[3]=5` → CHOOSE `6` | `[6]` | 2 | `[[1,1,6],[1,2,5],[1,7],[2,6]]` |
| 46 | `backtrack(5, 2)` → `i=5`: `7 > 2` → **BREAK** | `[6]` | 2 | `[[1,1,6],[1,2,5],[1,7],[2,6]]` |
| 47 | UN-CHOOSE (pop `6`) | `[]` | 8 | `[[1,1,6],[1,2,5],[1,7],[2,6]]` |
| 48 | `i=5`: `7 ≤ 8`, `7 ≠ candidates[4]=6` → CHOOSE `7` | `[7]` | 1 | `[[1,1,6],[1,2,5],[1,7],[2,6]]` |
| 49 | `backtrack(6, 1)` → `i=6`: `10 > 1` → **BREAK** | `[7]` | 1 | `[[1,1,6],[1,2,5],[1,7],[2,6]]` |
| 50 | UN-CHOOSE (pop `7`) | `[]` | 8 | `[[1,1,6],[1,2,5],[1,7],[2,6]]` |
| 51 | `i=6`: `10 > 8` → **BREAK** (ends `backtrack(0,8)`) — done | `[]` | 8 | `[[1,1,6],[1,2,5],[1,7],[2,6]]` |

Final `res = [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]` — matches
`assert cs2 == [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]`. Step 32 is the whole point of the duplicate guard:
without it, the second `1` at index `1` would restart the exact same subtree already built from index `0`
and every combination above would appear twice.

#### Mental model

- Same choose → explore → un-choose recursion as every other function in this notebook; only the
  recursive call's index (`i` vs `i + 1`) and one guard line change "reuse allowed" into "each slot once."
- `remaining` is the shrinking target, not a counter — recursion bottoms out at `remaining == 0` or a
  `break`, never by an explicit depth limit.
- Sorting is what makes `break` safe: once `candidates[i] > remaining`, every later candidate is too, so
  there's nothing left worth a `continue`.
- The duplicate skip compares **array neighbors within the same call** (`i > start`), not values already
  on `path` — that's why the second `1` is fine as the *first* pick (`i == start`, step 3) but is skipped
  as a *later* pick at the same level (step 32).

> **Why the rest of this notebook stays start-index / `used`-array:** include/exclude only works when
> the choice at each step is *binary* -- "is this one fixed array element in the answer or not". Once the
> choice becomes "which unused element goes next" (Permutations), "which of 4 directions" (Word Search),
> "how long is the next piece" (Palindrome Partitioning), "which letter for this digit" (Letter
> Combinations), or "which column" (N-Queens), there's no single element to include or exclude -- so these
> stay as start-index / `used`-array / direct-loop backtracking, same as before.

## 4. Permutations (LC 46, LC 47)

**Problem in plain language:**
- **Permutations (LC 46):** given a list of **distinct** numbers, return every possible **ordering**
  (arrangement) of them. Unlike subsets/combinations, order matters here -- `[1,2,3]` and `[3,2,1]` are
  different, separate answers.
- **Permutations II (LC 47):** same task, but the input can contain **duplicate** numbers. Return every
  **distinct** arrangement -- don't output the same ordering twice.

Order matters, so the start index is wrong here -- every level may pick **any** unused element. A `used`
array is the second piece of state, and it must be un-chosen too.

In [7]:
def permute(nums):                                   # LC 46
    out, path, used = [], [], [False] * len(nums)
    def backtrack():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue                                 # already on the path
            used[i] = True;  path.append(nums[i])        # CHOOSE (two pieces of state)
            backtrack()                                  # EXPLORE
            path.pop();      used[i] = False             # UN-CHOOSE (both pieces)
    backtrack()
    return out


def permute_unique(nums):                            # LC 47 -- input may contain duplicates
    nums = sorted(nums)
    out, path, used = [], [], [False] * len(nums)
    def backtrack():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            if i > 0 and nums[i] == nums[i - 1] and not used[i - 1]:
                continue                                 # twin to the left was already tried at this level
            used[i] = True;  path.append(nums[i])
            backtrack()
            path.pop();      used[i] = False
    backtrack()
    return out


p = permute([1, 2, 3])
print("permute([1,2,3]):", p)
assert len(p) == 6 and len({tuple(x) for x in p}) == 6     # 3! distinct orderings
assert [2, 1, 3] in p and [3, 2, 1] in p

pu = permute_unique([1, 1, 2])
print("permute_unique([1,1,2]):", pu)
assert pu == [[1, 1, 2], [1, 2, 1], [2, 1, 1]]

permute([1,2,3]): [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
permute_unique([1,1,2]): [[1, 1, 2], [1, 2, 1], [2, 1, 1]]


## 5. Word Search -- backtracking on a grid (LC 79)

**Problem in plain language:** you're given a 2D grid of letters and a target word. Starting from any
cell, can you spell out the word by moving one step at a time to a horizontally or vertically adjacent
cell, **without reusing the same cell twice** in one path? Return `True` if it's possible, `False`
otherwise.

The cell itself is the state: overwrite it with `"#"` on the way in, restore it on the way out. This variant
answers *yes / no*, so it returns `True` up the chain and `or` short-circuits at the first success.

In [8]:
def exist(board, word):                              # LC 79
    R, C = len(board), len(board[0])
    def dfs(r, c, k):                                    # k = index into `word` we must match at (r, c)
        if k == len(word):
            return True
        if not (0 <= r < R and 0 <= c < C) or board[r][c] != word[k]:
            return False
        saved, board[r][c] = board[r][c], "#"            # CHOOSE: mark the cell as on-path
        found = (dfs(r + 1, c, k + 1) or dfs(r - 1, c, k + 1) or
                 dfs(r, c + 1, k + 1) or dfs(r, c - 1, k + 1))
        board[r][c] = saved                              # UN-CHOOSE: always restore
        return found
    return any(dfs(r, c, 0) for r in range(R) for c in range(C))


board = [["A", "B", "C", "E"],
         ["S", "F", "C", "S"],
         ["A", "D", "E", "E"]]
snapshot = [row[:] for row in board]

for w, expected in [("ABCCED", True), ("SEE", True), ("ABCB", False)]:
    got = exist(board, w)
    print(f"exist({w!r}) = {got}")
    assert got == expected
assert board == snapshot          # every cell was restored, even along failed paths

exist('ABCCED') = True
exist('SEE') = True
exist('ABCB') = False


## 6. Palindrome Partitioning -- choose where to cut (LC 131)

**Problem in plain language:** given a string, cut it into pieces such that **every piece reads the same
forwards and backwards** (a palindrome). Return every possible way to make such a split.
Example: `"aab"` → `[["a","a","b"], ["aa","b"]]`.

The choice at each step is *how long the next piece is*. Only palindromic pieces are allowed, so the validity
check sits before the choose line and prunes everything else.

In [9]:
def partition(s):                                    # LC 131
    out, path = [], []
    def backtrack(start):
        if start == len(s):
            out.append(path[:]); return
        for end in range(start + 1, len(s) + 1):
            piece = s[start:end]
            if piece == piece[::-1]:                     # PRUNE: skip non-palindromic pieces
                path.append(piece); backtrack(end); path.pop()
    backtrack(0)
    return out


pp = partition("aab")
print("partition('aab'):", pp)
assert pp == [["a", "a", "b"], ["aa", "b"]]
assert all("".join(p) == "aab" for p in pp)      # every partition reassembles the input

partition('aab'): [['a', 'a', 'b'], ['aa', 'b']]


## 7. Letter Combinations of a Phone Number -- a Cartesian product (LC 17)

**Problem in plain language:** you're given a string of digits `2`-`9`, like an old phone keypad, where
each digit maps to a few letters (`2 -> "abc"`, `3 -> "def"`, ...). Return every possible letter
combination you could type by picking **one letter per digit**, keeping the digits' order.
Example: `"23"` → `["ad","ae","af","bd","be","bf","cd","ce","cf"]`.

No start index and no `used` set: every position is independent, and the tree's branching factor at depth `d`
is simply how many letters digit `d` maps to.

In [10]:
def letter_combinations(digits):                     # LC 17
    if not digits:
        return []
    phone = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl",
             "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}
    out, path = [], []
    def backtrack(i):
        if i == len(digits):
            out.append("".join(path)); return
        for ch in phone[digits[i]]:
            path.append(ch); backtrack(i + 1); path.pop()
    backtrack(0)
    return out


lc = letter_combinations("23")
print("letter_combinations('23'):", lc)
assert lc == ["ad", "ae", "af", "bd", "be", "bf", "cd", "ce", "cf"]
assert letter_combinations("") == []
assert len(letter_combinations("79")) == 4 * 4        # branching factors multiply

letter_combinations('23'): ['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']


## 8. N-Queens -- constraint satisfaction with O(1) pruning sets (LC 51)

**Problem in plain language:** place `n` chess queens on an `n x n` board so that **no two queens attack
each other** (no two share a row, a column, or a diagonal). Return every valid way to place them (each
answer shown as a board with `Q` for a queen and `.` for empty).

One queen per row; the choice is the column. Three sets make the safety check `O(1)`: every `\` diagonal shares
one `r - c`, every `/` diagonal shares one `r + c`. Four pieces of state to choose, four to un-choose.

In [11]:
def solve_n_queens(n):                               # LC 51
    out, cols, diag, anti = [], set(), set(), set()
    board = [["."] * n for _ in range(n)]
    def backtrack(r):
        if r == n:
            out.append(["".join(row) for row in board]); return
        for c in range(n):
            if c in cols or (r - c) in diag or (r + c) in anti:
                continue                                 # PRUNE: attacked by a queen above
            cols.add(c); diag.add(r - c); anti.add(r + c); board[r][c] = "Q"      # CHOOSE
            backtrack(r + 1)                                                      # EXPLORE
            cols.remove(c); diag.remove(r - c); anti.remove(r + c); board[r][c] = "."   # UN-CHOOSE
    backtrack(0)
    return out


sols = solve_n_queens(4)
print("4-queens solutions:")
for s in sols:
    print("   ", s)
assert sols == [[".Q..", "...Q", "Q...", "..Q."], ["..Q.", "Q...", "...Q", ".Q.."]]

counts = [len(solve_n_queens(n)) for n in range(1, 9)]
print("solution counts for n = 1..8:", counts)
assert counts == [1, 0, 0, 2, 10, 4, 40, 92]          # the classic sequence (OEIS A000170)

4-queens solutions:
    ['.Q..', '...Q', 'Q...', '..Q.']
    ['..Q.', 'Q...', '...Q', '.Q..']
solution counts for n = 1..8: [1, 0, 0, 2, 10, 4, 40, 92]


## 9. Measuring pruning -- how much does the sorted `break` save?

Same Combination Sum, three ways: no prune (only bail out once `remaining < 0`), `continue` past
over-large candidates, and `break` on the first over-large one. Count the recursive calls.

In [12]:
def combination_sum_counted(candidates, target, mode):
    """Same algorithm as combination_sum, with a call counter and a switchable prune."""
    candidates = sorted(candidates)
    out, path, calls = [], [], 0
    def backtrack(start, remaining):
        nonlocal calls
        calls += 1
        if remaining == 0:
            out.append(path[:]); return
        if remaining < 0:
            return                                       # the only guard in "none" mode
        for i in range(start, len(candidates)):
            if mode == "break" and candidates[i] > remaining:
                break
            if mode == "continue" and candidates[i] > remaining:
                continue
            path.append(candidates[i]); backtrack(i, remaining - candidates[i]); path.pop()
    backtrack(0, target)
    return calls, out


cands, target = [2, 3, 5, 7], 20
none_calls, none_out = combination_sum_counted(cands, target, "none")
cont_calls, cont_out = combination_sum_counted(cands, target, "continue")
brk_calls,  brk_out  = combination_sum_counted(cands, target, "break")
print(f"calls with no prune : {none_calls}")
print(f"calls with continue : {cont_calls}")
print(f"calls with break    : {brk_calls}")
print(f"answers found       : {len(brk_out)} (identical in all three modes)")
assert brk_calls <= cont_calls <= none_calls
assert brk_calls < none_calls                            # pruning genuinely removed work
assert none_out == cont_out == brk_out                   # ...without changing the answer
assert brk_out == combination_sum(cands, target)         # and matches the real function

calls with no prune : 246
calls with continue : 133
calls with break    : 133
answers found       : 18 (identical in all three modes)


## ✅ Recap

- **One template:** choose → explore → un-choose. Pruning checks go *before* the choose line; the un-choose
  line is an exact mirror of the choose line (every piece of state, not just `path`).
- **Order doesn't matter** (subsets, combinations, partitions) → a **start index**, only move forward.
  **Order matters** (permutations) → a **`used` array** and consider every element at every level.
- **Reuse allowed** → recurse with `i`; **each once** → `i + 1`.
- **Duplicate input values** → sort, then `if i > start and nums[i] == nums[i-1]: continue`.
- **Sorted + additive target** → `break`, not `continue`, on the first candidate that is too large.
- **Grids** → mark the cell itself, recurse 4 ways, restore. **Find-one** questions return `True` up the chain.
- **N-Queens** → three sets (`cols`, `r - c`, `r + c`) make the safety check `O(1)`.
- **Always record a copy** (`path[:]`). Time is (number of arrangements) × (cost to copy one).

**Coverage:** Blind 75 -- Combination Sum (39), Word Search (79). NeetCode 150 -- those two plus Subsets (78),
Subsets II (90), Combination Sum II (40), Permutations (46), Palindrome Partitioning (131), Letter Combinations (17),
N-Queens (51). Bonus: Combinations (77), Permutations II (47).

Prerequisites: [`04_Tree_Traversal`](../04_Tree_Traversal/README.md) (recursive DFS) and
[`07_Graph_Traversal`](../07_Graph_Traversal/README.md) (DFS on a grid).